# Two-opponent match

Both opponents follow 0.3 m behind the other. Stars show targets; short lines show forward direction.

Commands stay bounded, and the match stops at minimum separation. Reaching within 5 cm wins; simultaneous arrivals draw. The match times out at 30 seconds.

This tests a simple response model, not real flight dynamics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from IPython.display import HTML, display
from tracking import tracking_step, travel_direction, trailing_point

# Both opponents use the same settings and update at the same time.
period = 0.1
match_duration = 30.0
follow_distance = 0.3
reach_tolerance = 0.05
max_step = 0.05
min_separation = 0.1
response_time = 0.5
box_min = np.array([-1.0, -1.0, 0.15])
box_max = np.array([1.0, 1.0, 1.0])
positions = np.array([[-0.8, 0.8], [0.8, -0.8]])
commands = positions.copy()
heading = positions[::-1] - positions
heading /= np.linalg.norm(heading, axis=1)[:, None]
filtered_velocity = np.zeros((2, 2))
response_fraction = 1 - np.exp(-period / response_time)
statuses = ['tracking', 'tracking']
history = []
outcome = "Time limit — no winner"

for frame in range(round(match_duration / period) + 1):
    t = frame * period
    heading = np.array([travel_direction(filtered_velocity[i], heading[i])
                        for i in range(2)])
    # targets[i] is behind the OTHER opponent, not behind opponent i.
    targets = np.array([trailing_point(positions[1-i], heading[1-i], follow_distance)
                        for i in range(2)])
    history.append((t, positions.copy(), targets.copy(), heading.copy(), statuses.copy()))
    reached = np.linalg.norm(positions - targets, axis=1) <= reach_tolerance
    if reached.all():
        outcome = "Draw — both reached the trailing target"
        break
    if reached.any():
        outcome = f"Opponent {np.flatnonzero(reached)[0] + 1} reached the trailing target"
        break
    if frame == round(match_duration / period):
        break

    next_commands = []
    stop = False
    for i in range(2):
        next_command, _, too_close = tracking_step(
            commands[i], positions[i], positions[1-i], heading[1-i],
            follow_distance, box_min, box_max, max_step, min_separation)
        next_commands.append(next_command)
        statuses[i] = 'stop' if too_close else 'tracking'
        stop |= too_close
    if stop:
        outcome = "Separation stop — no winner"
        break
    commands = np.array(next_commands)
    next_positions = positions + response_fraction * (commands - positions)
    velocity = (next_positions - positions) / period
    filtered_velocity = 0.3 * velocity + 0.7 * filtered_velocity
    positions = next_positions

# Show only the animation, including the outcome on the last frame.
times = np.array([row[0] for row in history])
paths = np.array([row[1] for row in history])
targets = np.array([row[2] for row in history])
headings = np.array([row[3] for row in history])
modes = [row[4] for row in history]
assert np.all(np.isfinite(paths))
assert np.allclose(paths[:, 0], -paths[:, 1])  # Symmetric starts stay symmetric.
assert np.allclose(np.linalg.norm(paths[:, ::-1] - targets, axis=2), follow_distance)

fig, ax = plt.subplots(figsize=(6, 6))
ax.add_patch(Rectangle(box_min[:2], *(box_max[:2] - box_min[:2]),
                       fill=False, linestyle="--", color="gray"))
ax.set(xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), xlabel="World x (m)",
       ylabel="World y (m)", title="Opponent 1 vs opponent 2")
ax.set_aspect("equal")
ax.grid(alpha=0.2)
colors = ["tab:blue", "tab:orange"]
dots, trails, stars, links, directions = [], [], [], [], []
for i, color in enumerate(colors):
    dots.append(ax.plot([], [], "o", color=color, markersize=9,
                        label=f"Opponent {i+1}")[0])
    trails.append(ax.plot([], [], color=color, alpha=0.3)[0])
    stars.append(ax.plot([], [], "*", color=color, markersize=12,
                         label=f"Target for opponent {i+1}")[0])
    links.append(ax.plot([], [], ":", color=color, alpha=0.5)[0])
    directions.append(ax.plot([], [], color=color, linewidth=2)[0])
status = ax.text(0.02, 0.98, "", transform=ax.transAxes, fontsize=9, va="top",
                 bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
ax.legend(loc="lower left", fontsize=8)


def update(frame):
    for i in range(2):
        position = paths[frame, i]
        target = targets[frame, i]
        other = paths[frame, 1-i]
        dots[i].set_data([position[0]], [position[1]])
        trails[i].set_data(paths[:frame+1, i, 0], paths[:frame+1, i, 1])
        stars[i].set_data([target[0]], [target[1]])
        links[i].set_data([other[0], target[0]], [other[1], target[1]])
        forward = position + 0.12 * headings[frame, i]
        directions[i].set_data([position[0], forward[0]], [position[1], forward[1]])
    label = outcome if frame == len(times)-1 else " / ".join(modes[frame])
    status.set_text(f"{times[frame]:.1f} s | {label}")
    return dots + trails + stars + links + directions + [status]


# Hold the final frame briefly so the outcome is easy to read.
frames = list(range(len(times))) + [len(times)-1] * 20
animation = FuncAnimation(fig, update, frames=frames,
                          interval=period * 1000, repeat=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))
